# Feature Engineering

## Import required libraries

In [10]:
import pandas as pd
import re
import textstat

## Load Preprocessed Data

In [11]:
df = pd.read_csv('../preprocessing_data/processed_texts_final.csv')
df.head()

,cefr_folder,title,cefr_age_group,text,flesch_kincaid_score,smog_score,fk_estimated_uk_age_num,reading_band,lexicon_count,sentence_count,...,flesch_reading_ease,coleman_liau_index,automated_readability_index,dale_chall_readability_score,difficult_words,linsear_write_formula,gunning_fog,text_standard,spache_readability,reading_time
0,A1,airplane,6-8,"We are taking off, Margaret.\nWow. We are goin...",1.573182,5.985473,7,6-7,44,8,...,93.579773,1.227273,1.444545,7.139073,6,2.000000,3.109091,1st and 2nd grade,2.982682,2.76172
1,A1,alphabet_song,6-8,Do you know the Alphabet song?\nThe alphabet s...,2.347147,7.168622,7,6-7,39,8,...,86.917644,1.179487,1.538269,6.307531,4,1.937500,6.052564,1st and 2nd grade,2.849452,2.49730
2,A1,always_eating,6-8,Jane called Lisa. Lisa said she was eating. Sh...,0.980698,3.129100,6,6-7,123,22,...,97.989496,1.775610,0.933016,9.433890,2,1.777778,2.236364,0th and 1st grade,3.305367,7.50659
3,A1,angry_parent,6-8,You need to try harder in school!\nI am doing ...,2.603886,6.427356,8,8-12,83,12,...,88.713379,3.819277,2.911225,7.023422,8,2.791667,3.730522,2nd and 3rd grade,2.954009,5.40592
4,A1,animal_shelter,6-8,Today we go to the dog pound.\nWhat's that dad...,0.505119,3.129100,6,6-7,42,8,...,100.791964,-1.414286,-0.974286,5.776662,3,1.625000,2.100000,0th and 1st grade,2.193536,2.33571


## Create Word Count Feature

This calculates the total number of words in each text sample, adding this as the `word_count` feature. This provides a basic measure of text length, which is often useful in readability analysis.


In [12]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

## Calculate Average Word Length

This is the `average_word_length` feature, found by computing the mean number of characters per word in each text. This can help capture vocabulary complexity.


In [13]:
df['average_word_length'] = df['text'].apply(
    lambda x: sum(len(word) for word in str(x).split()) / len(str(x).split()) if len(str(x).split()) > 0 else 0
)

## Calculate Sentence Count

This is a simple regular expression, found by counting the number of sentence-ending punctuation marks (`.`, `!`, `?`) to estimate the number of sentences in each text (`sentence_count`).


In [14]:
df['sentence_count'] = df['text'].apply(lambda x: len(re.findall(r'[.!?]', str(x))))

## Add Complex Word Metrics

With the `textstat` library, this counts the number of complex words (those with three or more syllables) in each text.  
- `complex_word_count`: Total number of complex words.
- `percent_complex_words`: Proportion of complex words in the text.

These features capture aspects of vocabulary difficulty that go beyond simple word length.


In [15]:
df['complex_word_count'] = df['text'].apply(
    lambda x: sum([1 for word in str(x).split() if textstat.syllable_count(word) >= 3])
)
df['percent_complex_words'] = df['complex_word_count'] / df['word_count']

## Add Difficult Word Metrics

This uses `textstat`'s `difficult_words` function to count the number of words not found on standard “easy word” lists.
- `difficult_word_count`: Total number of difficult words.
- `percent_difficult_words`: Proportion of difficult words in the text.

These features help further quantify linguistic complexity from a learner's perspective.


In [16]:
df['difficult_word_count'] = df['text'].apply(lambda x: textstat.difficult_words(str(x)))
df['percent_difficult_words'] = df['difficult_word_count'] / df['word_count']

## Save Engineered Features

The resulting dataframe is saved, now containing both original and engineered features, to `feature_engineering.csv`. This file will be used for model training and evaluation in subsequent experiments.


In [17]:
df.to_csv('feature_engineering.csv', index=False)
# Check data frame is correct
df.head()

,cefr_folder,title,cefr_age_group,text,flesch_kincaid_score,smog_score,fk_estimated_uk_age_num,reading_band,lexicon_count,sentence_count,...,gunning_fog,text_standard,spache_readability,reading_time,word_count,average_word_length,complex_word_count,percent_complex_words,difficult_word_count,percent_difficult_words
0,A1,airplane,6-8,"We are taking off, Margaret.\nWow. We are goin...",1.573182,5.985473,7,6-7,44,11,...,3.109091,1st and 2nd grade,2.982682,2.76172,44,4.272727,2,0.045455,6,0.136364
1,A1,alphabet_song,6-8,Do you know the Alphabet song?\nThe alphabet s...,2.347147,7.168622,7,6-7,39,10,...,6.052564,1st and 2nd grade,2.849452,2.49730,39,4.358974,4,0.102564,4,0.102564
2,A1,always_eating,6-8,Jane called Lisa. Lisa said she was eating. Sh...,0.980698,3.129100,6,6-7,123,22,...,2.236364,0th and 1st grade,3.305367,7.50659,123,4.154472,0,0.000000,2,0.016260
3,A1,angry_parent,6-8,You need to try harder in school!\nI am doing ...,2.603886,6.427356,8,8-12,83,14,...,3.730522,2nd and 3rd grade,2.954009,5.40592,83,4.433735,4,0.048193,8,0.096386
4,A1,animal_shelter,6-8,Today we go to the dog pound.\nWhat's that dad...,0.505119,3.129100,6,6-7,42,10,...,2.100000,0th and 1st grade,2.193536,2.33571,42,3.785714,0,0.000000,3,0.071429


In [18]:
df = pd.read_csv('feature_engineering.csv')
df.head(10)

,cefr_folder,title,cefr_age_group,text,flesch_kincaid_score,smog_score,fk_estimated_uk_age_num,reading_band,lexicon_count,sentence_count,...,gunning_fog,text_standard,spache_readability,reading_time,word_count,average_word_length,complex_word_count,percent_complex_words,difficult_word_count,percent_difficult_words
0,A1,airplane,6-8,"We are taking off, Margaret.\nWow. We are goin...",1.573182,5.985473,7,6-7,44,11,...,3.109091,1st and 2nd grade,2.982682,2.76172,44,4.272727,2,0.045455,6,0.136364
1,A1,alphabet_song,6-8,Do you know the Alphabet song?\nThe alphabet s...,2.347147,7.168622,7,6-7,39,10,...,6.052564,1st and 2nd grade,2.849452,2.49730,39,4.358974,4,0.102564,4,0.102564
2,A1,always_eating,6-8,Jane called Lisa. Lisa said she was eating. Sh...,0.980698,3.129100,6,6-7,123,22,...,2.236364,0th and 1st grade,3.305367,7.50659,123,4.154472,0,0.000000,2,0.016260
3,A1,angry_parent,6-8,You need to try harder in school!\nI am doing ...,2.603886,6.427356,8,8-12,83,14,...,3.730522,2nd and 3rd grade,2.954009,5.40592,83,4.433735,4,0.048193,8,0.096386
4,A1,animal_shelter,6-8,Today we go to the dog pound.\nWhat's that dad...,0.505119,3.129100,6,6-7,42,10,...,2.100000,0th and 1st grade,2.193536,2.33571,42,3.785714,0,0.000000,3,0.071429
5,A1,an_application_form,6-8,Library card application\nFirst name\tMIKE\nLa...,10.917167,12.161745,16,15-16,75,5,...,11.766667,11th and 12th grade,5.661417,6.12573,84,4.964286,10,0.119048,16,0.190476
6,A1,an_email_to_book_a_hotel,6-8,o: info@ascot-hotel.co.uk\nFrom: David Mathews...,5.919316,8.167269,11,8-12,117,12,...,7.251282,6th and 7th grade,3.995077,7.82977,117,4.555556,7,0.059829,15,0.128205
7,A1,an_email_to_confirm_an_appointment,6-8,"From: Arina Marat, HR Assistant\nTo: Jane Clar...",5.953696,9.188382,11,8-12,92,8,...,8.078261,5th and 6th grade,4.236587,6.09635,92,4.510870,9,0.097826,14,0.152174
8,A1,an_email_to_congratulate_a_colleague,6-8,To: Jon\nFrom: Sue\nSubject: Great presentatio...,3.943230,7.554174,9,8-12,87,9,...,5.778851,3rd and 4th grade,3.153056,5.77317,87,4.517241,6,0.068966,9,0.103448
9,A1,apple_for_the_teacher,6-8,Andrew was in the third grade. He loved his te...,1.475835,5.148861,6,6-7,101,16,...,2.525000,2nd and 3rd grade,2.325102,6.56643,101,4.425743,2,0.019802,2,0.019802
